# T2.4 – DBRepo View Definitions

This notebook defines and creates DBRepo views for the Vienna Weather Wet-Month Prediction experiment.

The notebook creates a DBRepo-compatible view over the `weather_measurement_v2` table. The full SQL view definitions, including the joined feature view and train/validation/test splits, are documented in `sql/create_views.sql`.

In [1]:
import os
from getpass import getpass

from dbrepo.RestClient import RestClient
from dbrepo.api.dto import QueryDefinition

In [2]:
ENDPOINT = "https://test.dbrepo.tuwien.ac.at"

USERNAME = "azra1558"
PASSWORD = "Katalizator1558!"

DATABASE_ID = "a181cad5-4bdb-48b2-937e-3e75293f6a7b"

TABLE_IDS = {
    "weather_measurement_v2": "3674fea3-a7be-4dfe-8356-bc692bd1ff6c",
    "time_dimension": "fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde",
    "station": "ab02386c-e27c-4c1f-a27d-93034ce3fa79"
}

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("Connected to DBRepo.")

Connected to DBRepo.


In [3]:
tables = client.get_tables(DATABASE_ID)

for table in tables:
    print(table.name, table.id)

weather_measurement_v2 3674fea3-a7be-4dfe-8356-bc692bd1ff6c
weather_measurement 631c878e-1f39-47a0-be38-0b3c0e2733c8
time_dimension fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde
station ab02386c-e27c-4c1f-a27d-93034ce3fa79


In [4]:
def get_view_by_name(client, database_id, view_name):
    views = client.get_views(database_id)
    for view in views:
        if view.name == view_name:
            return view
    return None

In [5]:
weather_feature_columns = [
    "weather_measurement_v2.measurement_id",
    "weather_measurement_v2.station_num",
    "weather_measurement_v2.time_id",
    "weather_measurement_v2.t_mean_c",
    "weather_measurement_v2.t_max_c",
    "weather_measurement_v2.t_min_c",
    "weather_measurement_v2.mean_t_max_c",
    "weather_measurement_v2.mean_t_min_c",
    "weather_measurement_v2.p_mean_hpa",
    "weather_measurement_v2.p_max_hpa",
    "weather_measurement_v2.p_min_hpa",
    "weather_measurement_v2.precp_sum_mm",
    "weather_measurement_v2.num_precp_01",
    "weather_measurement_v2.rel_hum_pct",
    "weather_measurement_v2.rel_hum_max_pct",
    "weather_measurement_v2.rel_hum_min_pct",
    "weather_measurement_v2.wind_vel_ms",
    "weather_measurement_v2.wind_vel_max_ms",
    "weather_measurement_v2.num_wind_vel60",
    "weather_measurement_v2.sun_h",
    "weather_measurement_v2.num_clear",
    "weather_measurement_v2.num_cloud",
    "weather_measurement_v2.num_frost",
    "weather_measurement_v2.num_ice",
    "weather_measurement_v2.num_summer",
    "weather_measurement_v2.num_heat",
]

view_name = "weather_measurement_v2_features"

existing_view = get_view_by_name(client, DATABASE_ID, view_name)

if existing_view is not None:
    print("View already exists:", existing_view.name, existing_view.id)
else:
    query = QueryDefinition(
        datasources=["weather_measurement_v2"],
        columns=weather_feature_columns
    )

    view = client.create_view(
        database_id=DATABASE_ID,
        name=view_name,
        query=query,
        is_public=False,
        is_schema_public=True
    )

    print("Created view:", view.name, view.id)

Created view: weather_measurement_v2_features 77031135-c567-482f-9606-ef9f220b0446


In [7]:
views = client.get_views(DATABASE_ID)

print("DBRepo views:")
for view in views:
    row_count = client.get_view_data_count(DATABASE_ID, view.id)
    print(f"- {view.name}: {view.id} ({row_count} rows)")

DBRepo views:
- weather_measurement_v2_features: 77031135-c567-482f-9606-ef9f220b0446 (1845 rows)


## Note on full SQL views

The DBRepo Python API was used to create the DBRepo-compatible view
`weather_measurement_features`.

The full SQL view definitions are documented in
`sql/create_views.sql`. These include the joined feature view, the train,
validation and test views, and a monthly precipitation summary view.

The views currently return zero rows because the actual data loading is handled
later in T2.5.